In [1]:
from pyspark.sql import SparkSession, DataFrame

# Create Spark session
Start or reuse a Spark session that the rest of the pipeline will share.

In [2]:
def create_spark_session(app_name: str, master: str = "local[*]") -> SparkSession:
    """Create a Spark session, or reuse one that is already running.

    Args:
        app_name: Name shown for this application in Spark.
        master: Where Spark runs. Defaults to local, using all CPU cores.

    Returns:
        The active SparkSession.
    """
    return SparkSession.builder.appName(app_name).master(master).getOrCreate()

In [3]:
#test run
spark = create_spark_session("file_ingestion")
spark

------------------------

#  Read file into DataFrame 
Read one validated file with Spark, using a configurable delimiter and header option.

In [4]:
# imports for building explicit schema
from pyspark.sql.types import StructType,StructField,StringType,IntegerType

schema = StructType(
                    [
                     StructField("id",IntegerType(),nullable=True),
                     StructField("name",StringType(),nullable=True),
                     StructField("address",StringType(),nullable=True),
                     StructField("amount",IntegerType(),nullable=True)
                    ]
                    )

In [5]:
def read_file_to_dataframe(spark: SparkSession,
                           file_path: str, 
                           schema: StructType, 
                           delimiter: str = ",",
                           header: bool = True,
                           mode: str = "FAILFAST") -> DataFrame: 
    """Read a delimited text file into a Spark DataFrame.
 
    Args:
        spark: Active Spark session.
        file_path: Full path of the file to read.
        schema: Column names and types the DataFrame must follow.
        delimiter: Character that separates the columns. Defaults to a comma.
        header: True if the first line is a header row, so Spark skips it.
        mode: How to handle bad values. Defaults to FAILFAST, which raises
            an error instead of silently turning a bad value into null.
 
    Returns:
        A DataFrame with one row per data line. Column names come from the schema.
    """
    
    return(spark.read.options(header=header, sep=delimiter, mode=mode).schema(schema).csv(file_path))

In [6]:
test_goodvalue = read_file_to_dataframe(spark,"../raw_files/john_london_20261012_093000.txt",schema)
test_goodvalue.show()
test_goodvalue.printSchema()

+---+----+-------+------+
| id|name|address|amount|
+---+----+-------+------+
|  1|John| London|   250|
|  2|Emma| London|   400|
|  3|Liam| London|   150|
+---+----+-------+------+

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- address: string (nullable = true)
 |-- amount: integer (nullable = true)



In [7]:
test_badvalue = read_file_to_dataframe(spark, "../raw_files/test_badvalue_20261009_100000.txt",schema)
test_badvalue.show()
test_badvalue.printSchema()

Py4JJavaError: An error occurred while calling o52.showString.
: org.apache.spark.SparkException: [FAILED_READ_FILE.NO_HINT] Encountered error while reading file file:///C:/Users/v/Multi%20Layer%20File%20Ingestion/raw_files/test_badvalue_20261009_100000.txt.  SQLSTATE: KD001
	at org.apache.spark.sql.errors.QueryExecutionErrors$.cannotReadFilesError(QueryExecutionErrors.scala:911)
	at org.apache.spark.sql.execution.datasources.v2.FileDataSourceV2$.attachFilePath(FileDataSourceV2.scala:142)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:141)
	at scala.collection.Iterator$$anon$9.hasNext(Iterator.scala:593)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:43)
	at org.apache.spark.sql.execution.WholeStageCodegenEvaluatorFactory$WholeStageCodegenPartitionEvaluator$$anon$1.hasNext